In [0]:
drop table if exists max_nfs_promocao; 
create temporary table max_nfs_promocao as 
(select
  purchase_receipt_detail_access_key_content as nf,
  trim(promotion_name) as promocao,
  max(cast(update_date as timestamp)) as data_maxima_nf
from prod.silver_shopping.users_cashback_purchases_receipts
where 1 = 1
  and cast(date_trunc('day', timestampadd(hour, -3, cast(create_date as timestamp))) as date) >= '2024-01-01'
  -- and purchase_receipt_detail_access_key_content = '35240144480747000160590013876700062972720711'
group by all)
    

num_affected_rows,num_inserted_rows


In [0]:
SET STATEMENT_TIMEOUT = 1800;

drop table if exists nfs_campanha;
create temporary table nfs_campanha as (
select 
  *,
  case when item_id is not null then 'validas' else 'invalidas' end as flag
from (
select distinct 
  date_trunc('hour', timestampadd(hour, -3, cast(scan_date as timestamp))) as datahora_criacao,
  cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date as timestamp))) as date) as data_criacao,
  date_trunc('hour', timestampadd(hour, -3, coalesce(try_to_timestamp(regexp_replace(regexp_replace(
    purchase_receipt_read_detail_purchase_receipt_details_issue_date, ';', ':'),'-\\d{2}:\\d{2}$', ''),'dd/MM/yyyy HH:mm:ss'),
      try_to_timestamp(purchase_receipt_read_detail_purchase_receipt_details_issue_date, 'yyyy-MM-dd HH:mm:ss')
      )
    )
  ) AS datahora_emissao,
  cast(date_trunc('day', timestampadd(hour, -3, coalesce(try_to_timestamp(regexp_replace(regexp_replace(
    purchase_receipt_read_detail_purchase_receipt_details_issue_date, ';', ':'),'-\\d{2}:\\d{2}$', ''),'dd/MM/yyyy HH:mm:ss'),
      try_to_timestamp(purchase_receipt_read_detail_purchase_receipt_details_issue_date, 'yyyy-MM-dd HH:mm:ss')
      )
    )
  ) AS date) as data_emissao,
  date_format(cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date as timestamp))) as date), 'yyyyMM') as anomes,
  cast(update_date as timestamp) as data_atualizacao,
  purchase_receipt_detail_access_key_content as nf,
  purchase_receipt_read_detail_purchase_receipt_details_items_detail.value:_id::string as item_id,
  user_document as cpf,
  purchase_receipt_read_detail_purchase_receipt_details_address_detail_uf as uf_ec,
  purchase_receipt_read_detail_purchase_receipt_details_company_detail_cnpj as cnpj_ec,
  trim(promotion_name) as promocao,
  status,
  reject_reason as motivo_rejeicao,
  cashback_promotion_id,
  case
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('campanhalanamentopontomais') then '1'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('diadasmesvr') then '2'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('juntosemisturadosagosto2024') then '3'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('semanadoconsumidor') then '4' 
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr025') then '5'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr100novembro') then '6'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr1000') then '7'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr100') then '8'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackporenviodenf') then '9'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiosabril') then '10'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('petrobrasabril') then '11'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('r100pornf') then '12' 
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('superpromo') then '13'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiossetembro') then '14' 
  else cashback_promotion_id end as promotion_id,
  cast(purchase_receipt_read_detail_purchase_receipt_details_cost_detail_product_service_total_cost_in_cents as float) / 100 as custo_nf,
  cast(purchase_receipt_read_detail_purchase_receipt_details_total_purchase_in_cents as float) / 100 as valor_nf,
  cast(purchase_receipt_read_detail_purchase_receipt_details_cost_detail_item_total_discount_cost_in_cents as float) / 100 as desconto_nf,
  round(purchase_receipt_read_detail_purchase_receipt_details_items_detail.value:amount::double, 2) as qtd_produtos
from prod.silver_shopping.users_cashback_purchases_receipts
  , lateral variant_explode_outer(purchase_receipt_read_detail_purchase_receipt_details_items_detail) as purchase_receipt_read_detail_purchase_receipt_details_items_detail
where 1 = 1
  and cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date as timestamp))) as date) >= '2024-01-01'
)
)

key,value
STATEMENT_TIMEOUT,1800


num_affected_rows,num_inserted_rows


In [0]:
drop table if exists nfs_resumed;
create temporary table nfs_resumed as (
select distinct 
  a.*,
  b.purchase_receipt_detail_qrcode_content
from 
  (select 
    max(processing_details.value:createDate::timestamp) as max_status,
    purchase_receipt_detail_access_key_content
  from prod.silver_shopping.users_cashback_purchases_receipts_resumed
    , lateral variant_explode_outer(processing_details) as processing_details
  where 1 = 1
  group by all) a
  left join 
  (select 
    *,
    processing_details.value:createDate::timestamp as data_status
  from prod.silver_shopping.users_cashback_purchases_receipts_resumed
    , lateral variant_explode_outer(processing_details) as processing_details
  where 1 = 1) b on 
    b.data_status = a.max_status
    and b.purchase_receipt_detail_access_key_content = a.purchase_receipt_detail_access_key_content
where 1 = 1
group by all
)

num_affected_rows,num_inserted_rows


In [0]:
SET STATEMENT_TIMEOUT = 1800;

drop table if exists nfs_campanha_join;
create temporary table nfs_campanha_join as (
select
  b.datahora_criacao,
  b.data_criacao,
  b.anomes,
  b.datahora_emissao,
  b.data_emissao,
  a.nf,
  count(distinct b.item_id) as qtd_itens,
  b.cpf,
  b.uf_ec,
  case when b.uf_ec in ('AM', 'RR', 'AP', 'AC', 'RO', 'PA', 'TO') then 'NORTE'
    when b.uf_ec in ('MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA') then 'NORDESTE'
    when b.uf_ec in ('MT', 'GO', 'DF', 'MS') then 'CENTRO-OESTE'
    when b.uf_ec in ('MG', 'SP', 'ES', 'RJ') then 'SUDESTE'
    when b.uf_ec in ('PR', 'SC', 'RS') then 'SUL'
  else 'INDEFINIDO' end as regiao,
  b.cnpj_ec,
  b.promocao,
  case when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%ajinomoto%' then 'Ajinomoto'
    when trim(b.promocao) = 'Café União - Fevereiro' then 'União'
    when trim(b.promocao) like '%Plasútil%' then 'Plasútil'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%arcor%' then 'Arcor'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%guarani%' then 'Guarani - Tereos'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%brilux%' then 'Ray Mundo da Fonte'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%uniao%' then 'União'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%tereos%' then 'Guarani - Tereos'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%pontomais%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%raymundo%' then 'Ray Mundo da Fonte'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%marilan%' then 'Marilan'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%bolocasasuica%' then 'Ray Mundo da Fonte'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%condor%' then 'Condor'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%zinho%' then 'Zinho'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%kibon%' then 'Unilever'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%cheetos%' then 'PepsiCo'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%doritos%' then 'PepsiCo'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%lays%' then 'PepsiCo'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%ruffles%' then 'PepsiCo'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%correios%' then 'Campanha Correios'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%danone%' then 'Danone'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%didasmaesvr%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%frutap%' then 'Frutap'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%havanna%' then 'Havanna'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%huggies%' then 'Kimberly-Clark'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%juntosemisturados%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%popcorners%' then 'PepsiCo'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%petrobras%' then 'Campanha Petrobras'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%plasutil%' then 'Plasútil'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%kibon%' then 'Unilever'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%sorteio%' then 'Sorteio VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%r100pornf%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%santamassa%' then 'Santa Massa'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%semanadoconsumidor%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%sorteionotasficalpremiada%' then 'Sorteio VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) = 'vrcashback' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackporenviodenf%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackr100%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackr025%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackr1000%' then 'Campanha VR'
    when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%superpromo%' then 'Campanha VR' else null 
  end as fabricante,
  b.custo_nf,
  b.desconto_nf,
  b.valor_nf,
  sum(b.qtd_produtos) as qtd_produtos,
  b.flag,
  b.status,
  b.motivo_rejeicao,
  b.cashback_promotion_id,
  b.promotion_id,
  c.purchase_receipt_detail_qrcode_content
from max_nfs_promocao a
left join nfs_campanha b on 
  b.promocao = a.promocao
  and b.nf = a.nf
  and b.data_atualizacao = a.data_maxima_nf
left join nfs_resumed c on 
  c.purchase_receipt_detail_access_key_content = a.nf
where 1 = 1
group by all
)

key,value
STATEMENT_TIMEOUT,1800


num_affected_rows,num_inserted_rows


In [0]:
-- create or replace temporary view max_nfs_promocao_invalidas as (
-- select
--   purchase_receipt_detail_access_key_content as nf,
--   max(cast(update_date_date as timestamp)) as data_maxima_nf,
--   trim(promotion_name) as promocao
-- from prod.silver_vr_shopping.users_cashback_purchases_receipts_nfs_invalidas
-- where 1 = 1
--   and cast(date_trunc('day', timestampadd(hour, -3, cast(create_date_date as timestamp))) as date) >= '2024-01-01'
-- group by all
-- )

In [0]:
-- create or replace temporary view nfs_campanha_invalidas as (
-- select distinct 
--   date_trunc('hour', timestampadd(hour, -3, cast(scan_date_date as timestamp))) as datahora_criacao,
--   cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date_date as timestamp))) as date) as data_criacao,
--   date_format(cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date_date as timestamp))) as date), 'yyyyMM') as anomes,
--   cast(update_date_date as timestamp) as data_atualizacao,
--   date_trunc('hour', timestampadd(hour, -3, coalesce(try_to_timestamp(regexp_replace(regexp_replace(
--     purchase_receipt_read_detail_purchase_receipt_details_issue_date, ';', ':'),'-\\d{2}:\\d{2}$', ''),'dd/MM/yyyy HH:mm:ss'),
--       try_to_timestamp(purchase_receipt_read_detail_purchase_receipt_details_issue_date, 'yyyy-MM-dd HH:mm:ss')
--       )
--     )
--   ) AS datahora_emissao,
--   cast(date_trunc('day', timestampadd(hour, -3, coalesce(try_to_timestamp(regexp_replace(regexp_replace(
--     purchase_receipt_read_detail_purchase_receipt_details_issue_date, ';', ':'),'-\\d{2}:\\d{2}$', ''),'dd/MM/yyyy HH:mm:ss'),
--       try_to_timestamp(purchase_receipt_read_detail_purchase_receipt_details_issue_date, 'yyyy-MM-dd HH:mm:ss')
--       )
--     )
--   ) AS date) as data_emissao,
--   purchase_receipt_detail_access_key_content as nf,
--   null as item_id,
--   user_document as cpf,
--   purchase_receipt_read_detail_purchase_receipt_details_address_detail_uf as uf_ec,
--   purchase_receipt_read_detail_purchase_receipt_details_company_detail_cnpj as cnpj_ec,
--   trim(promotion_name) as promocao,
--   status,
--   reject_reason as motivo_rejeicao,
--   cashback_promotion_id_oid,
--   purchase_receipt_detail_qrcode_content,
--   case
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('campanhalanamentopontomais') then '1'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('diadasmesvr') then '2'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('juntosemisturadosagosto2024') then '3'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('semanadoconsumidor') then '4' 
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr025') then '5'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr100novembro') then '6'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr1000') then '7'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr100') then '8'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackporenviodenf') then '9'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiosabril') then '10'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('petrobrasabril') then '11'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('r100pornf') then '12'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('superpromo') then '13'
--     when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiossetembro') then '14' else cashback_promotion_id_oid end as promotion_id,
--   cast(purchase_receipt_read_detail_purchase_receipt_details_cost_detail_product_service_total_cost_in_cents as float) / 100 as custo_nf,
--   cast(purchase_receipt_read_detail_purchase_receipt_details_total_purchase_in_cents as float) / 100 as valor_nf,
--   cast(purchase_receipt_read_detail_purchase_receipt_details_cost_detail_item_total_discount_cost_in_cents as float) / 100 as desconto_nf,
--   round(cast(purchase_receipt_read_detail_purchase_receipt_details_items_detail_amount as float), 2) as qtd_produtos,
--   case when cast(scan_date_date as date) >= current_date() - 45 and status not in ('REJECTED', 'ERROR') then 'em processamento' else 'invalidas' end as flag
-- from prod.silver_vr_shopping.users_cashback_purchases_receipts_nfs_invalidas
-- where 1 = 1
--   and cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date_date as timestamp))) as date) >= '2024-01-01'
-- )

In [0]:
-- create or replace temporary view nfs_campanha_invalidas_join as (
-- select 
--   b.datahora_criacao,
--   b.data_criacao,
--   b.anomes,
--   b.datahora_emissao,
--   b.data_emissao,
--   a.nf,
--   b.item_id as qtd_itens,
--   b.cpf,
--   b.uf_ec,
--   case when b.uf_ec in ('AM', 'RR', 'AP', 'AC', 'RO', 'PA', 'TO') then 'NORTE'
--     when b.uf_ec in ('MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA') then 'NORDESTE'
--     when b.uf_ec in ('MT', 'GO', 'DF', 'MS') then 'CENTRO-OESTE'
--     when b.uf_ec in ('MG', 'SP', 'ES', 'RJ') then 'SUDESTE'
--     when b.uf_ec in ('PR', 'SC', 'RS') then 'SUL'
--   else 'INDEFINIDO' end as regiao,
--   b.cnpj_ec,
--   b.promocao,
--   case when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%ajinomoto%' then 'Ajinomoto'
--     when trim(b.promocao) = 'Café União - Fevereiro' then 'União'
--     when trim(b.promocao) like '%Plasútil%' then 'Plasútil'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%arcor%' then 'Arcor'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%guarani%' then 'Guarani - Tereos'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%brilux%' then 'Ray Mundo da Fonte'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%uniao%' then 'União'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%tereos%' then 'Guarani - Tereos'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%pontomais%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%raymundo%' then 'Ray Mundo da Fonte'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%marilan%' then 'Marilan'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%bolocasasuica%' then 'Ray Mundo da Fonte'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%condor%' then 'Condor'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%zinho%' then 'Zinho'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%kibon%' then 'Unilever'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%cheetos%' then 'PepsiCo'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%doritos%' then 'PepsiCo'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%lays%' then 'PepsiCo'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%ruffles%' then 'PepsiCo'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%correios%' then 'Campanha Correios'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%danone%' then 'Danone'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%didasmaesvr%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%frutap%' then 'Frutap'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%havanna%' then 'Havanna'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%huggies%' then 'Kimberly-Clark'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%juntosemisturados%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%popcorners%' then 'PepsiCo'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%petrobras%' then 'Campanha Petrobras'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%plasutil%' then 'Plasútil'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%kibon%' then 'Unilever'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%sorteio%' then 'Sorteio VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%r100pornf%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%santamassa%' then 'Santa Massa'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%semanadoconsumidor%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%sorteionotasficalpremiada%' then 'Sorteio VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) = 'vrcashback' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackporenviodenf%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackr100%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackr025%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%vrcashbackr1000%' then 'Campanha VR'
--     when lower(regexp_replace(b.promocao,'[^0-9a-zA-Z]','')) like '%superpromo%' then 'Campanha VR' else null 
--   end as fabricante,
--   b.custo_nf,
--   b.desconto_nf,
--   b.valor_nf,
--   sum(b.qtd_produtos) as qtd_produtos,
--   b.flag,
--   b.status,
--   b.motivo_rejeicao,
--   b.cashback_promotion_id_oid,
--   b.promotion_id,
--   b.purchase_receipt_detail_qrcode_content
-- from max_nfs_promocao_invalidas a
-- left join nfs_campanha_invalidas b on
--   b.nf = a.nf
--   and b.data_atualizacao = a.data_maxima_nf
--   and b.promocao = a.promocao
-- where 1 = 1
-- group by all
-- )

In [0]:
-- create or replace temporary view nfs_campanha_union as (
-- select * from nfs_campanha_join
-- union all 
-- select * from nfs_campanha_invalidas_join
-- )

In [0]:
-- SET STATEMENT_TIMEOUT = 3600;

-- select * from nfs_campanha_union
-- where 1 = 1
--   and fabricante = 'Campanha Correios'
--   and data_criacao >= '2025-09-01'
-- limit 10

In [0]:
drop table if exists produtos_promocao;
create temporary table produtos_promocao as (
select * from prod.silver_shopping.cashbacks_engines_promotions_products
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists nfs_cashback; 
create temporary table nfs_cashback as (
select distinct 
  date_trunc('hour', timestampadd(hour, -3, cast(scan_date as timestamp))) as datahora_criacao,
  cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date as timestamp))) as date) as data_criacao,
  date_trunc('hour', timestampadd(hour, -3, cast(engine_processing_result_transactions_details_transaction_date as timestamp))) as datahora_cashback,
  cast(date_trunc('day', timestampadd(hour, -3, cast(engine_processing_result_transactions_details_transaction_date as timestamp))) as date) as data_cashback,
  purchase_receipt_detail_access_key_content as nf,
  cashback_promotion_id,
  trim(promotion_name) as promocao,
  case
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('campanhalanamentopontomais') then '1'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('diadasmesvr') then '2'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('juntosemisturadosagosto2024') then '3'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('semanadoconsumidor') then '4' 
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr025') then '5'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr100novembro') then '6'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr1000') then '7'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackr100') then '8'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('vrcashbackporenviodenf') then '9'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiosabril') then '10'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('petrobrasabril') then '11'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('r100pornf') then '12'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('superpromo') then '13'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiossetembro') then '14' else cashback_promotion_id end as promotion_id,
  case
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in 
      ('campanhalanamentopontomais', 'diadasmesvr', 'juntosemisturadosagosto2024', 'semanadoconsumidor', 
      'vrcashbackr025', 'vrcashbackr100novembro', 'vrcashbackr1000', 'vrcashbackr100', 'vrcashbackporenviodenf', 'r100pornf', 'superpromo') then 'Campanha VR'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiosabril') then 'Campanha Correios'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('correiossetembro') then 'Campanha Correios'
    when lower(regexp_replace(promotion_name,'[^0-9a-zA-Z]','')) in ('petrobrasabril') then 'Campanha Petrobras' else trim(promotion_name) 
  end as fabricante_aux,
  user_document,
  float(engine_processing_result_cashback_reward_calculated_total_amount_in_cents) / 100 as cashback_entregue,
  case when purchase_receipt_read_detail_purchase_receipt_details_items_detail.value:identifiedEan::string is null then purchase_receipt_read_detail_purchase_receipt_details_items_detail.value:businessEan::string else purchase_receipt_read_detail_purchase_receipt_details_items_detail.value:identifiedEan::string end as ean
from prod.silver_shopping.users_cashback_purchases_receipts
  , lateral variant_explode_outer(purchase_receipt_read_detail_purchase_receipt_details_items_detail) as purchase_receipt_read_detail_purchase_receipt_details_items_detail
where 1 = 1 
  and engine_processing_result_cashback_reward_result_status = 'CREDITED'
  and cast(date_trunc('day', timestampadd(hour, -3, cast(scan_date as timestamp))) as date) >= '2024-01-01'
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists nfs_cashback_produtos_promocao;
create temporary table nfs_cashback_produtos_promocao as (
select distinct 
  a.datahora_criacao as datetime_criacao,
  a.data_criacao,
  max(a.data_cashback) over (partition by a.user_document) as maxima_data_cashback,
  min(a.data_cashback) over (partition by a.user_document) as primeira_data_cashback,
  a.datahora_cashback as datetime_cashback,
  a.data_cashback as data_cashback,
  a.nf,
  case when b.manufacturer in (' Ajinomoto®','Aajinomoto ','Ajinomoto','Ajinomoto ®','Ajinomoto®') then 'Ajinomoto'
    when b.manufacturer in ('Guarani - Tereos','Guarani -Tereos', 'Tereos', 'Tereos Guarani') then 'Guarani - Tereos'
    when b.manufacturer in ('PepsiCo','PepsiCo - Cheetos','PepsiCo - PopCorners','Pepsico') then 'PepsiCo'
    when b.manufacturer in ('Santa Massa','Snata Massa') then 'Santa Massa' 
    when b.manufacturer in ('HAvanna','Havanna') then 'Havanna'
    when b.manufacturer in ('frutap','Frutap') then 'Frutap'
    when b.manufacturer in ('Unilever','Kibon') then 'Unilever' 
    when a.fabricante_aux in ('Campanha VR') then 'Campanha VR'
    when a.fabricante_aux in ('Campanha Correios') then 'Campanha Correios'
    when a.fabricante_aux in ('Campanha Petrobras') then 'Campanha Petrobras'
    when a.fabricante_aux in ('Zinho ') then 'Zinho'
  else b.manufacturer end as manufacturer,
  -- case when trim(b.brand) is not null then b.brand else a.promotion_name end as brand,
  -- a.promotion_name,
  a.cashback_promotion_id,
  a.promotion_id,
  a.user_document as cpf,
  a.cashback_entregue
from nfs_cashback a 
left join produtos_promocao b on
  b.cashback_engine_promotion_id = a.cashback_promotion_id
  and b.ean = a.ean
where 1 = 1
)

num_affected_rows,num_inserted_rows


In [0]:

drop table if exists nfs_cashback_produtos_promocao_drillup;
create temporary table nfs_cashback_produtos_promocao_drillup as (
select 
  datetime_criacao,
  data_criacao,
  date_format(data_criacao, 'yyyyMM') as anomes,
  maxima_data_cashback,
  primeira_data_cashback,
  date_format(maxima_data_cashback, 'yyyyMM') as anomes_maximo_cashback,
  date_format(primeira_data_cashback, 'yyyyMM') as anomes_primeiro_cashback,
  datetime_cashback,
  data_cashback,
  nf,
  manufacturer,
  cpf,
  cashback_promotion_id,
  promotion_id,
  cashback_entregue
from nfs_cashback_produtos_promocao
where 1 = 1
  and manufacturer is not null
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists nfs_campanha_cashback;
create temporary table nfs_campanha_cashback as (
select 
  a.data_criacao,
  a.datahora_criacao,
  min(a.data_criacao) over (partition by a.cpf) as primeira_data_envio,
  max(a.data_criacao) over (partition by a.cpf) as maxima_data_envio,
  a.anomes,
  a.datahora_emissao,
  a.data_emissao,
  b.primeira_data_cashback,
  b.maxima_data_cashback,
  b.anomes_primeiro_cashback,
  b.anomes_maximo_cashback,
  b.datetime_cashback,
  b.data_cashback,
  a.nf,
  a.qtd_itens,
  a.cpf,
  a.uf_ec,
  a.regiao,
  a.cnpj_ec,
  a.promocao,
  a.custo_nf,
  a.desconto_nf,
  a.valor_nf,
  a.qtd_produtos,
  a.flag,
  a.status,
  a.motivo_rejeicao,
  a.promotion_id,
  b.cashback_promotion_id,
  case when trim(b.manufacturer) is null then trim(a.fabricante) else trim(b.manufacturer) end as manufacturer,
  case when b.nf is null then 0 else b.cashback_entregue end as cashback_entregue,
  case when a.purchase_receipt_detail_qrcode_content is not null then 1 else 0 end as qr_code_escaneado
from nfs_campanha_join a
left join nfs_cashback_produtos_promocao_drillup b on
  b.nf = a.nf
  and b.promotion_id = a.promotion_id
where 1 = 1
  and a.promotion_id is not null
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists dim_cadastro;
create temporary table dim_cadastro as (
select * from prod.gold_vr_cliente_trabalhador.dim_cadastro
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists dim_persona;
create temporary table dim_persona as (
select * from prod.gold_vr_cliente_trabalhador.dim_persona
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists dim_ecs;
create temporary table dim_ecs as (
SELECT distinct 
  srk_ec_cadastro,
  num_cnpj
FROM prod.gold_vr_cliente_ec.dim_cadastro
)

num_affected_rows,num_inserted_rows


In [0]:
drop table if exists nfs_ofertas_campanhas;
create temporary table nfs_ofertas_campanhas as (
select 
  c.*,
  date_format(c.primeira_data_envio, 'yyyyMM') as anomes_primeiro_envio,
  date_format(c.maxima_data_envio, 'yyyyMM') as anomes_maximo_envio,
  e.srk_trabalhador_cadastro,
  g.srk_ec_cadastro,
  f.dsc_persona,
  f.cod_persona
from nfs_campanha_cashback c
left join dim_cadastro e on 
  e.num_cpf = c.cpf
left join dim_persona f on 
  f.cod_persona = e.cod_persona
left join dim_ecs g on
  g.num_cnpj = c.cnpj_ec
)

num_affected_rows,num_inserted_rows


In [0]:
SET STATEMENT_TIMEOUT = 1800;

drop table if exists nfs_resumed_campanha;
create temporary table nfs_resumed_campanha as (
SELECT
  * except(rn, uf_ec),
  dense_rank() OVER (ORDER BY nf asc) as nf_index,
  CASE SUBSTRING(nf, 1, 2)
    WHEN '11' THEN 'RO'
    WHEN '12' THEN 'AC'
    WHEN '13' THEN 'AM'
    WHEN '14' THEN 'RR'
    WHEN '15' THEN 'PA'
    WHEN '16' THEN 'AP'
    WHEN '17' THEN 'TO'
    WHEN '21' THEN 'MA'
    WHEN '22' THEN 'PI'
    WHEN '23' THEN 'CE'
    WHEN '24' THEN 'RN'
    WHEN '25' THEN 'PB'
    WHEN '26' THEN 'PE'
    WHEN '27' THEN 'AL'
    WHEN '28' THEN 'SE'
    WHEN '29' THEN 'BA'
    WHEN '31' THEN 'MG'
    WHEN '32' THEN 'ES'
    WHEN '33' THEN 'RJ'
    WHEN '35' THEN 'SP'
    WHEN '41' THEN 'PR'
    WHEN '42' THEN 'SC'
    WHEN '43' THEN 'RS'
    WHEN '50' THEN 'MS'
    WHEN '51' THEN 'MT'
    WHEN '52' THEN 'GO'
    WHEN '53' THEN 'DF'
    ELSE 'Desconhecido'
  END AS uf_ec,
  case when business_unit.datalab.valida_chave_nfe(nf) <> 'valido' then 'invalido' else 'valido' end as flag_funcao
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY nf, promocao ORDER BY CASE WHEN flag = 'invalidas' THEN 1 ELSE 0 END) as rn
  FROM nfs_ofertas_campanhas
) ranked_nfs
WHERE 1 = 1
-- and nf = '43241245151273000175650020000759591697540981'
  and rn = 1
)

key,value
STATEMENT_TIMEOUT,1800


num_affected_rows,num_inserted_rows


In [0]:
-- SET STATEMENT_TIMEOUT = 36000;
drop table if exists nfs_ofertas_campanhas_full;
create temporary table nfs_ofertas_campanhas_full as (
select 
  a.datahora_criacao,
  a.nf,
  b.* except(b.datahora_criacao, b.nf)
from 
(select 
  max(datahora_criacao) as datahora_criacao,
  nf
from nfs_resumed_campanha
group by all
) a 
left join nfs_resumed_campanha b on
  b.datahora_criacao = a.datahora_criacao
  and b.nf = a.nf
)

num_affected_rows,num_inserted_rows


In [0]:
SET STATEMENT_TIMEOUT = 1800;

create or replace table business_unit.informacoes_trabalhador.tb_nfs_ofertas_campanhas as (
select * from nfs_ofertas_campanhas_full
)

key,value
STATEMENT_TIMEOUT,1800


num_affected_rows,num_inserted_rows


In [0]:
select 
  nf,
  count(distinct data_criacao)
from business_unit.informacoes_trabalhador.tb_nfs_ofertas_campanhas
where 1 = 1
group by all
having count(distinct data_criacao) > 1

nf,count(DISTINCT data_criacao)


In [0]:
select max(datahora_criacao) from business_unit.informacoes_trabalhador.tb_nfs_ofertas_campanhas

max(datahora_criacao)
2026-04-13T22:00:00.000Z


In [0]:
create or replace table business_unit.informacoes_trabalhador.tb_nfs_ofertas_status as (
WITH nf_prioridades AS (
    SELECT
        nf_index,
        status,
        data_criacao,
        uf_ec,
        CASE 
            WHEN status = 'IN_ANALYSIS' THEN 1
            WHEN status = 'AWAITING_INTEGRATION' THEN 2
            WHEN status = 'AWAITING_INTEGRATION_RESULT' THEN 3
            WHEN status = 'AWAITING_ENGINE_PROCESSING' THEN 4
            WHEN status = 'AWAITING_CREDIT_PROCESSING' THEN 5
            WHEN status IN ('REJECTED', 'COMPLETED', 'ERROR') THEN 6
            ELSE 999
        END AS prioridade_status
    FROM business_unit.informacoes_trabalhador.tb_nfs_ofertas_campanhas
    where 1 = 1
      and data_criacao >= '2025-01-01'
)
SELECT nf_index, status, prioridade_status, data_criacao, uf_ec
FROM (
    SELECT
        nf_index,
        status,
        prioridade_status,
        data_criacao,
        uf_ec,
        ROW_NUMBER() OVER (PARTITION BY nf_index ORDER BY prioridade_status ASC, data_criacao DESC) AS rn
    FROM nf_prioridades
) ranked
WHERE rn = 1
)

num_affected_rows,num_inserted_rows
